# Integration tests de OP-07 `read_trips`

Notebook orientado a verificar el comportamiento integrado de la operación pública OP-07 `read_trips()`.

Estos tests no prueban helpers internos de forma aislada. En su lugar, preparan artefactos formales de trips, principalmente mediante `write_trips()` como setup, y luego verifican que `read_trips()` reconstruya correctamente un `TripDataset` desde disco.

Objetivo:
- probar lectura formal de bundles `.golondrina`;
- verificar reconstrucción de `TripDataset`;
- verificar resolución de path con sufijo `.golondrina`;
- verificar uso de sidecar `trips.metadata.json`;
- verificar reconstrucción de `schema` y `schema_effective`;
- verificar preservación de provenance y correspondencias;
- verificar política post-read `metadata["is_validated"] = False`;
- verificar evento `read_trips`;
- cubrir backend Parquet y backend Feather;
- probar fallas públicas relevantes de OP-07.

Nota:
- `write_trips()` se usa solo para generar artefactos válidos de setup.
- Los tests exclusivos de OP-06 quedan fuera de este notebook.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- resolución robusta del root del repositorio;
- imports generales;
- imports del módulo;
- helpers de testing reutilizables;
- carpeta local visible para artefactos.

### 0.1 Resolución del repositorio

Qué prepara: permite ejecutar el notebook desde distintas ubicaciones dentro del repositorio, agregando `src/` y el root al `sys.path` cuando corresponde.

Los artefactos de prueba se escribirán en una carpeta local relativa al directorio actual del notebook.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 
print("NOTEBOOK_ROOT =", NOTEBOOK_ROOT)
print("REPO_ROOT     =", REPO_ROOT)

NOTEBOOK_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips
REPO_ROOT     = C:\projects\pylondrina


### 0.2 Imports generales

Qué prepara: imports base, utilidades de filesystem, pandas y PyArrow para inspeccionar artefactos físicos cuando sea necesario.

In [2]:
import copy
import json
import shutil

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather

### 0.3 Imports del módulo

Qué prepara: operaciones públicas necesarias para construir fixtures ricas y probar OP-07.

Importante: `write_trips()` se importa solo para crear artefactos válidos de setup. La operación bajo prueba es `read_trips()`.

In [3]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.datasets import TripDataset

from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips

from pylondrina.errors import ExportError

from pylondrina.io.trips import (
    write_trips,
    read_trips,
    WriteTripsOptions,
    ReadTripsOptions,
)

### 0.4 Import del generador sintético

Qué prepara: acceso al generador usado para construir fixtures ricas de integración.

In [4]:
from scripts.synthetic_data.base_generator import generate_synthetic_trip_dataframe

### 0.5 Helpers de testing reutilizables

In [5]:
def show_ok(label: str):
    print(f"OK - {label}")


def get_issue_codes(issues):
    return [issue.code for issue in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente {code}. Codes actuales: {codes}"


def clone_tripdataset(trips: TripDataset) -> TripDataset:
    return copy.deepcopy(trips)


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def load_sidecar(artifact_dir: Path) -> dict:
    sidecar_path = artifact_dir / "trips.metadata.json"
    assert sidecar_path.exists(), f"No existe sidecar: {sidecar_path}"
    return json.loads(sidecar_path.read_text(encoding="utf-8"))


def write_sidecar(artifact_dir: Path, payload: dict) -> None:
    sidecar_path = artifact_dir / "trips.metadata.json"
    sidecar_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def assert_data_equivalent(left: pd.DataFrame, right: pd.DataFrame):
    pd.testing.assert_frame_equal(
        left.reset_index(drop=True),
        right.reset_index(drop=True),
        check_dtype=False,
        check_categorical=False,
    )


def artifact_data_filename(storage_format: str) -> str:
    if storage_format == "parquet":
        return "trips.parquet"
    if storage_format == "feather":
        return "trips.feather"
    raise ValueError(f"storage_format no soportado: {storage_format!r}")


def artifact_data_file_path(artifact_dir: Path, storage_format: str) -> Path:
    return artifact_dir / artifact_data_filename(storage_format)


def selected_categorical_columns(df: pd.DataFrame | None = None) -> list[str]:
    cols = [
        "mode",
        "purpose",
        "day_type",
        "time_period",
        "user_gender",
        "user_age_group",
        "income_quintile",
    ]
    if df is None:
        return cols
    return [c for c in cols if c in df.columns]


def series_as_string_with_na(series: pd.Series) -> pd.Series:
    return series.astype("string")


def observed_non_null_values(series: pd.Series) -> set[str]:
    s = series_as_string_with_na(series).dropna()
    return set(s.tolist())

### 0.6 Configuración de display

In [6]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

show_ok("Imports y helpers cargados")

OK - Imports y helpers cargados


### 0.7 Carpeta visible de integración

Qué prepara: carpeta local para artefactos de persistencia generados por los tests.

La carpeta se crea en el directorio actual del notebook y se reinicia al ejecutar esta celda.

In [7]:
IT_ROOT = Path("./tmp_op07_read_trips_integration").resolve()


def reset_it_root() -> Path:
    if IT_ROOT.exists():
        shutil.rmtree(IT_ROOT)
    IT_ROOT.mkdir(parents=True, exist_ok=True)
    return IT_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = IT_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_it_root()
print("IT_ROOT =", root)
show_ok("Sección 0 lista")

IT_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips\tmp_op07_read_trips_integration
OK - Sección 0 lista


## Sección 1. Fixtures ricas reutilizables

Esta sección construye una fixture integrada realista usando el generador sintético, `import_trips_from_dataframe()` y `validate_trips()`.

La idea es que los tests de OP-07 no lean un dataframe mínimo artificial, sino artefactos que provienen de un `TripDataset` ya construido y validado mediante el flujo público previo del módulo.

### 1.1 Constantes de schema para la fixture rica

Qué prepara: campos requeridos, campos base y dominios canónicos suficientes para importación, validación y persistencia.

In [8]:
REQUIRED_FIELDS_ORDER = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_h3_index",
    "destination_h3_index",
    "origin_time_utc",
    "destination_time_utc",
    "trip_id",
    "movement_seq",
]

REQUIRED_FIELD_DTYPES = {
    "movement_id": "string",
    "user_id": "string",
    "origin_longitude": "float",
    "origin_latitude": "float",
    "destination_longitude": "float",
    "destination_latitude": "float",
    "origin_h3_index": "string",
    "destination_h3_index": "string",
    "origin_time_utc": "datetime",
    "destination_time_utc": "datetime",
    "trip_id": "string",
    "movement_seq": "int",
}

BASE_FIELD_DTYPES = {
    "origin_municipality": "string",
    "destination_municipality": "string",
    "timezone_offset_min": "int",
    "origin_time_local_hhmm": "string",
    "destination_time_local_hhmm": "string",
    "trip_weight": "float",
    "mode_sequence": "string",
    "mode": "categorical",
    "purpose": "categorical",
    "day_type": "categorical",
    "time_period": "categorical",
    "user_gender": "categorical",
    "user_age_group": "categorical",
    "income_quintile": "categorical",
}

CANONICAL_DOMAINS = {
    "mode": [
        "walk", "bicycle", "scooter", "motorcycle", "car",
        "taxi", "ride_hailing", "bus", "metro", "train", "other",
    ],
    "purpose": [
        "home", "work", "education", "shopping", "errand",
        "health", "leisure", "transfer", "other",
    ],
    "day_type": ["weekday", "weekend", "holiday"],
    "time_period": ["night", "morning", "midday", "afternoon", "evening"],
    "user_gender": ["female", "male", "other", "unknown"],
    "user_age_group": ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
    "income_quintile": ["1", "2", "3", "4", "5", "unknown"],
}

RICH_BASE_FIELDS = [
    "origin_municipality",
    "destination_municipality",
    "timezone_offset_min",
    "origin_time_local_hhmm",
    "destination_time_local_hhmm",
    "trip_weight",
    "mode_sequence",
    "mode",
    "purpose",
    "day_type",
    "time_period",
    "user_gender",
    "user_age_group",
    "income_quintile",
]

RICH_EXTRA_COLUMNS = [
    "activity_status",
    "education_level",
    "travel_time_bucket",
    "season",
    "fare_payment_type",
    "bike_lane_usage",
    "home_tenure",
]

### 1.2 Builder de schema rica

Qué prepara: un `TripSchema` suficientemente rico para importación y validación de datasets sintéticos realistas.

In [9]:
def make_field(name: str, dtype: str, *, required: bool = False, domain: DomainSpec | None = None) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=None,
        domain=domain,
    )


def make_rich_trip_schema() -> TripSchema:
    fields = {}

    for field_name in REQUIRED_FIELDS_ORDER:
        fields[field_name] = make_field(
            field_name,
            REQUIRED_FIELD_DTYPES[field_name],
            required=True,
        )

    for field_name, dtype_name in BASE_FIELD_DTYPES.items():
        domain = None
        if dtype_name == "categorical":
            domain = DomainSpec(values=CANONICAL_DOMAINS[field_name], extendable=True)

        fields[field_name] = make_field(
            field_name,
            dtype_name,
            required=False,
            domain=domain,
        )

    return TripSchema(
        version="1.1",
        fields=fields,
        required=list(REQUIRED_FIELDS_ORDER),
        semantic_rules=None,
    )

### 1.3 Builder de source dataframe rica

Qué prepara: un dataframe de entrada más rico que los smoke tests, usando generador sintético e incluyendo campos base y columnas extra.

In [10]:
def build_rich_source_dataframe(seed: int = 20260404, filas: int = 180) -> pd.DataFrame:
    df = generate_synthetic_trip_dataframe(
        filas=filas,
        seed=seed,
        duplicate_mode="none",
        tier_temporal="tier_1",
        tier1_datetime_format="utc_string_z",
        coord_format="numeric",
        h3_mode="provided_valid",
        trip_structure="multistage",
        max_movements_per_trip=3,
        base_fields=RICH_BASE_FIELDS,
        extra_value_domains={
            "mode": ["canon"],
            "purpose": ["canon"],
            "day_type": ["canon"],
            "time_period": ["canon"],
            "user_gender": ["canon"],
            "user_age_group": ["canon"],
            "income_quintile": ["canon"],
        },
        extra_columns=RICH_EXTRA_COLUMNS,
        null_ratio={
            "origin_municipality": 0.03,
            "destination_municipality": 0.03,
        },
    )
    return df

### 1.4 Fixtures base de integración

Qué prepara:
- `trip_schema_snapshot_small`;
- `tripdataset_canonical_small`;
- `tripdataset_unvalidated_small`;
- `tripdataset_validated_small`.

Aunque se llamen `small`, estas fixtures son suficientemente ricas para integración: pasan por importación pública, validación pública, metadata, schema efectivo, dominios y eventos previos.

In [11]:
trip_schema_snapshot_small = make_rich_trip_schema()

source_df_rich = build_rich_source_dataframe(filas=180)

tripdataset_canonical_small, canonical_import_report = import_trips_from_dataframe(
    source_df_rich,
    trip_schema_snapshot_small,
    source_name="synthetic_rich_trips",
    options=ImportOptions(
        keep_extra_fields=True,
        selected_fields=None,
        strict=False,
        strict_domains=False,
        single_stage=False,
        source_timezone=None,
    ),
    provenance={
        "source": {"name": "synthetic_generator", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
        "notes": ["fixture de integración OP-07 read_trips"],
    },
    h3_resolution=8,
)

assert canonical_import_report.ok is True
assert tripdataset_canonical_small.metadata["is_validated"] is False

tripdataset_unvalidated_small = clone_tripdataset(tripdataset_canonical_small)

tripdataset_validated_small = clone_tripdataset(tripdataset_canonical_small)
validated_report_fixture = validate_trips(
    tripdataset_validated_small,
    options=ValidationOptions(
        strict=False,
        validate_domains="full",
    ),
)

assert validated_report_fixture.ok is True
assert tripdataset_validated_small.metadata["is_validated"] is True

print("source_df_rich.shape =", source_df_rich.shape)
print("canonical shape     =", tripdataset_canonical_small.data.shape)
print("validated shape     =", tripdataset_validated_small.data.shape)
show_ok("Sección 1 lista")

source_df_rich.shape = (180, 33)
canonical shape     = (180, 33)
validated shape     = (180, 33)
OK - Sección 1 lista


### 1.5 Helper para materializar artefactos válidos desde la API pública

Qué prepara: setup reusable para tests de lectura.

`write_trips()` se usa aquí para producir bundles formales realistas. El subject de los tests sigue siendo `read_trips()`.

In [12]:
def write_valid_artifact_with_backend(
    case_dir: Path,
    artifact_name: str = "artifact_bundle",
    *,
    storage_format: str = "parquet",
    parquet_compression: str = "snappy",
    feather_compression: str = "lz4",
) -> tuple[TripDataset, Path, Path, object]:
    trips = clone_tripdataset(tripdataset_validated_small)
    base_path = case_dir / artifact_name

    report = write_trips(
        trips,
        base_path,
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format=storage_format,
            parquet_compression=parquet_compression,
            feather_compression=feather_compression,
            normalize_artifact_dir=True,
        ),
    )

    artifact_dir = case_dir / f"{artifact_name}.golondrina"
    data_path = artifact_data_file_path(artifact_dir, storage_format)

    assert report.ok is True
    assert artifact_dir.exists()
    assert data_path.exists()
    assert (artifact_dir / "trips.metadata.json").exists()

    return trips, artifact_dir, data_path, report

## Sección 2. Integration tests de `read_trips()`

Esta sección prueba la operación pública OP-07 usando artefactos formales reales.

Los tests verifican reconstrucción de dataset, sidecar, metadata, eventos, schema, schema_effective, backend físico y políticas de recuperación.

### Test 1 - read feliz Parquet usando path sin sufijo y schema desde metadata

Qué prueba:
- fallback amigable a `.golondrina` en `read_trips()`;
- reconstrucción de schema desde snapshot del sidecar cuando `options.schema=None`;
- preservación de identidad;
- regla post-read `is_validated=False`;
- append del evento `read_trips`;
- equivalencia lógica de datos.

In [13]:
case_dir = make_case_dir("test_01_read_parquet_fallback_metadata_schema")

written_trips, artifact_dir, data_path, write_report = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="parquet",
    parquet_compression="snappy",
)

loaded, read_report = read_trips(
    case_dir / "bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert Path(read_report.parameters["path"]) == artifact_dir
assert read_report.parameters["schema"]["source"] == "metadata"
assert read_report.summary["path"] == str(artifact_dir)
assert read_report.summary["schema_source"] == "metadata"
assert read_report.summary["storage_format"] == "parquet"

assert loaded.metadata["is_validated"] is False
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

assert loaded.metadata["dataset_id"] == written_trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == written_trips.metadata["artifact_id"]

assert loaded.metadata["events"][-1]["op"] == "read_trips"
assert loaded.metadata["events"][-1]["parameters"] == read_report.parameters
assert loaded.metadata["events"][-1]["summary"] == read_report.summary

assert_data_equivalent(loaded.data, written_trips.data)

display(loaded.data.head())
display(read_report.issues)
display(read_report.summary)
show_ok("Test 1 - read feliz Parquet con fallback y schema desde metadata")

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.300000,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c51b69fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.450000,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c57293fffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented


[Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})]

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_01_read_parquet_fallback_metadata_schema\\bundle.golondrina',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_318b7159-85f7-494d-bca5-b88223a1472b',
 'artifact_id_status': 'loaded'}

OK - Test 1 - read feliz Parquet con fallback y schema desde metadata


### Test 2 - read feliz Feather usando path sin sufijo y schema desde metadata

Qué prueba:
- mismo contrato observable que el caso Parquet;
- backend Feather resuelto desde sidecar;
- lectura de `trips.feather`;
- preservación de identidad;
- `is_validated=False` post-read;
- evento `read_trips`.

In [14]:
case_dir = make_case_dir("test_02_read_feather_fallback_metadata_schema")

written_trips, artifact_dir, data_path, write_report = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="feather",
    feather_compression="lz4",
)

loaded, read_report = read_trips(
    case_dir / "bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert data_path.name == "trips.feather"
assert Path(read_report.parameters["path"]) == artifact_dir
assert read_report.parameters["schema"]["source"] == "metadata"
assert read_report.summary["schema_source"] == "metadata"
assert read_report.summary["storage_format"] == "feather"

assert loaded.metadata["is_validated"] is False
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

assert loaded.metadata["dataset_id"] == written_trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == written_trips.metadata["artifact_id"]

assert loaded.metadata["events"][-1]["op"] == "read_trips"
assert loaded.metadata["events"][-1]["parameters"] == read_report.parameters
assert loaded.metadata["events"][-1]["summary"] == read_report.summary

assert_data_equivalent(loaded.data, written_trips.data)

display(loaded.data.head())
display(read_report.summary)
show_ok("Test 2 - read feliz Feather con fallback y schema desde metadata")

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.300000,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c51b69fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.450000,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c57293fffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented


{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_02_read_feather_fallback_metadata_schema\\bundle.golondrina',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_0947827d-d8f7-40dd-afdd-32e787f9f5ac',
 'artifact_id_status': 'loaded'}

OK - Test 2 - read feliz Feather con fallback y schema desde metadata


### Test 3 - read con schema explícito y precedencia sobre metadata

Qué prueba:
- `ReadTripsOptions.schema` tiene precedencia sobre el snapshot persistido;
- un mismatch observable queda reportado con `READ.SCHEMA.MISMATCH`;
- en `strict=False`, la lectura sigue siendo recuperable.

In [15]:
case_dir = make_case_dir("test_03_read_schema_precedence_parquet")

written_trips, artifact_dir, _, _ = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="parquet",
    parquet_compression="snappy",
)

schema_override = copy.deepcopy(trip_schema_snapshot_small)
schema_override.version = "1.1-override"

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=schema_override,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True
assert loaded.schema.version == "1.1-override"
assert read_report.parameters["schema"]["source"] == "options"
assert read_report.parameters["schema"]["version"] == "1.1-override"
assert read_report.summary["schema_source"] == "options"
assert read_report.summary["schema_mismatch"] is True

assert_issue_present(read_report.issues, "READ.SCHEMA.MISMATCH")
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

display(read_report.issues)
display(read_report.summary)
show_ok("Test 3 - read con schema explícito y precedencia")

[Issue(level='warning', code='READ.SCHEMA.MISMATCH', message='El schema provisto por options no coincide con el snapshot persistido en el sidecar; se usará options.schema según precedencia.', field=None, source_field=None, row_count=None, details={'schema_source': 'options', 'schema_mismatch': True, 'version_options': '1.1-override', 'version_metadata': '1.1', 'required_diff': [], 'fields_diff_sample': [], 'fields_diff_total': 0, 'action': 'use_options_schema'}),
 Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})]

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_03_read_schema_precedence_parquet\\bundle.golondrina',
 'storage_format': 'parquet',
 'schema_source': 'options',
 'schema_mismatch': True,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_5a715dc4-cf06-42cd-a249-ae014dcfd9f2',
 'artifact_id_status': 'loaded'}

OK - Test 3 - read con schema explícito y precedencia


### Test 4 - read con schema explícito y `strict=True` ante mismatch

Qué prueba:
- el mismo mismatch de schema deja de ser recuperable cuando `strict=True`;
- `read_trips()` debe abortar con `ExportError`.

In [16]:
case_dir = make_case_dir("test_04_read_schema_precedence_strict_true")

_, artifact_dir, _, _ = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="parquet",
    parquet_compression="snappy",
)

schema_override = copy.deepcopy(trip_schema_snapshot_small)
schema_override.version = "1.1-strict-mismatch"

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=schema_override,
            strict=True,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.SCHEMA.MISMATCH"

display(raised)
show_ok("Test 4 - read strict=True fatal por schema mismatch")

ExportError(message='El schema provisto por options no coincide con el snapshot persistido en el sidecar; se usará options.schema según precedencia.', code='READ.SCHEMA.MISMATCH', details={'schema_source': 'options', 'schema_mismatch': True, 'version_options': '1.1-strict-mismatch', 'version_metadata': '1.1', 'required_diff': [], 'fields_diff_sample': [], 'fields_diff_total': 0, 'action': 'use_options_schema'}, issue=Issue(level='warning', code='READ.SCHEMA.MISMATCH', message='El schema provisto por options no coincide con el snapshot persistido en el sidecar; se usará options.schema según precedencia.', field=None, source_field=None, row_count=None, details={'schema_source': 'options', 'schema_mismatch': True, 'version_options': '1.1-strict-mismatch', 'version_metadata': '1.1', 'required_diff': [], 'fields_diff_sample': [], 'fields_diff_total': 0, 'action': 'use_options_schema'}), issues=(Issue(level='warning', code='READ.SCHEMA.MISMATCH', message='El schema provisto por options no co

OK - Test 4 - read strict=True fatal por schema mismatch


### Test 5 - round-trip básico completo Parquet

Qué prueba:
- pipeline público `write_trips() -> read_trips()` sobre Parquet;
- fidelidad lógica del dataframe;
- preservación de schema, schema_effective, provenance y correspondencias;
- preservación de identidad;
- política de eventos.

In [17]:
case_dir = make_case_dir("test_05_roundtrip_basic_parquet")

trips_original = clone_tripdataset(tripdataset_validated_small)
data_original = trips_original.data.copy(deep=True)
schema_original = copy.deepcopy(trips_original.schema)
schema_effective_original = copy.deepcopy(trips_original.schema_effective)
provenance_original = copy.deepcopy(trips_original.provenance)
field_corr_original = copy.deepcopy(trips_original.field_correspondence)
value_corr_original = copy.deepcopy(trips_original.value_correspondence)

write_report = write_trips(
    trips_original,
    case_dir / "roundtrip_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

loaded, read_report = read_trips(
    case_dir / "roundtrip_bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert loaded.metadata["dataset_id"] == trips_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips_original.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False

assert_data_equivalent(loaded.data, data_original)

assert loaded.schema.to_dict() == schema_original.to_dict()
assert loaded.schema_effective.to_dict() == schema_effective_original.to_dict()
assert loaded.provenance == provenance_original
assert loaded.field_correspondence == field_corr_original
assert loaded.value_correspondence == value_corr_original

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_trips" in ops_loaded
assert ops_loaded[-1] == "read_trips"

display(write_report.summary)
display(read_report.summary)
show_ok("Test 5 - round-trip básico completo Parquet")

{'n_rows': 180,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_05_roundtrip_basic_parquet\\roundtrip_bundle.golondrina',
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'artifact_id': 'art_855ba44d-da8e-4506-ab5a-8979757c6855',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_05_roundtrip_basic_parquet\\roundtrip_bundle.golondrina',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_855ba44d-da8e-4506-ab5a-8979757c6855',
 'artifact_id_status': 'loaded'}

OK - Test 5 - round-trip básico completo Parquet


### Test 6 - round-trip básico completo Feather

Qué prueba:
- pipeline público `write_trips() -> read_trips()` sobre Feather;
- fidelidad lógica del dataframe;
- preservación de schema, schema_effective, provenance y correspondencias;
- preservación de identidad;
- política de eventos.

In [18]:
case_dir = make_case_dir("test_06_roundtrip_basic_feather")

trips_original = clone_tripdataset(tripdataset_validated_small)
data_original = trips_original.data.copy(deep=True)
schema_original = copy.deepcopy(trips_original.schema)
schema_effective_original = copy.deepcopy(trips_original.schema_effective)
provenance_original = copy.deepcopy(trips_original.provenance)
field_corr_original = copy.deepcopy(trips_original.field_correspondence)
value_corr_original = copy.deepcopy(trips_original.value_correspondence)

write_report = write_trips(
    trips_original,
    case_dir / "roundtrip_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=True,
    ),
)

loaded, read_report = read_trips(
    case_dir / "roundtrip_bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert loaded.metadata["dataset_id"] == trips_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips_original.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False

assert_data_equivalent(loaded.data, data_original)

assert loaded.schema.to_dict() == schema_original.to_dict()
assert loaded.schema_effective.to_dict() == schema_effective_original.to_dict()
assert loaded.provenance == provenance_original
assert loaded.field_correspondence == field_corr_original
assert loaded.value_correspondence == value_corr_original

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_trips" in ops_loaded
assert ops_loaded[-1] == "read_trips"

display(write_report.summary)
display(read_report.summary)
show_ok("Test 6 - round-trip básico completo Feather")

{'n_rows': 180,
 'files_written': ['trips.feather', 'trips.metadata.json'],
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_06_roundtrip_basic_feather\\roundtrip_bundle.golondrina',
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'artifact_id': 'art_9bf92d28-9bb2-4131-a859-f7d30da9ba74',
 'dataset_id_status': 'preserved',
 'storage_format': 'feather'}

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_06_roundtrip_basic_feather\\roundtrip_bundle.golondrina',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_9bf92d28-9bb2-4131-a859-f7d30da9ba74',
 'artifact_id_status': 'loaded'}

OK - Test 6 - round-trip básico completo Feather


### Test 7 - read fatal por layout inválido Parquet sin sidecar

Qué prueba:
- `read_trips()` no acepta un archivo tabular suelto como artefacto formal;
- el sidecar `trips.metadata.json` es obligatorio.

In [19]:
case_dir = make_case_dir("test_07_read_fatal_missing_sidecar_parquet")
artifact_dir = case_dir / "broken_bundle.golondrina"
artifact_dir.mkdir(parents=True, exist_ok=True)

tripdataset_validated_small.data.to_parquet(
    artifact_dir / "trips.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.MISSING_SIDECAR"

display(raised)
show_ok("Test 7 - read fatal por sidecar faltante Parquet")

ExportError(message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", code='READ.LAYOUT.MISSING_SIDECAR', details={'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_07_read_fatal_missing_sidecar_parquet\\broken_bundle.golondrina', 'resolved_path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_07_read_fatal_missing_sidecar_parquet\\broken_bundle.golondrina', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.parquet'], 'files_present_total': 1, 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.MISSING_SIDECAR', message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", field=None, source_field=None, row_count=None, details={'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_07_read_fatal_missing_sidecar_parquet\\broken_bun

OK - Test 7 - read fatal por sidecar faltante Parquet


### Test 8 - read fatal por layout inválido Feather sin sidecar

Qué prueba:
- misma precondición formal del test anterior;
- ahora con `trips.feather` como archivo tabular aislado.

In [20]:
case_dir = make_case_dir("test_08_read_fatal_missing_sidecar_feather")
artifact_dir = case_dir / "broken_bundle.golondrina"
artifact_dir.mkdir(parents=True, exist_ok=True)

table = pa.Table.from_pandas(tripdataset_validated_small.data, preserve_index=False)
feather.write_feather(
    table,
    artifact_dir / "trips.feather",
    compression="lz4",
    version=2,
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.MISSING_SIDECAR"

display(raised)
show_ok("Test 8 - read fatal por sidecar faltante Feather")

ExportError(message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", code='READ.LAYOUT.MISSING_SIDECAR', details={'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_08_read_fatal_missing_sidecar_feather\\broken_bundle.golondrina', 'resolved_path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_08_read_fatal_missing_sidecar_feather\\broken_bundle.golondrina', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.feather'], 'files_present_total': 1, 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.MISSING_SIDECAR', message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", field=None, source_field=None, row_count=None, details={'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_08_read_fatal_missing_sidecar_feather\\broken_bun

OK - Test 8 - read fatal por sidecar faltante Feather


### Test 9 - read fatal por sidecar legacy

Qué prueba:
- `read_trips()` debe rechazar `metadata.json` legacy cuando falta `trips.metadata.json`;
- evita confundir persistencia formal vigente con artefactos antiguos o informales.

In [21]:
case_dir = make_case_dir("test_09_read_fatal_legacy_sidecar")
artifact_dir = case_dir / "legacy_bundle.golondrina"
artifact_dir.mkdir(parents=True, exist_ok=True)

tripdataset_validated_small.data.to_parquet(
    artifact_dir / "trips.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

(artifact_dir / "metadata.json").write_text(
    json.dumps({"legacy": True}, ensure_ascii=False),
    encoding="utf-8",
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.LEGACY_SIDECAR_DETECTED"

display(raised)
show_ok("Test 9 - read fatal por sidecar legacy")

ExportError(message="Se detectó un sidecar legacy 'metadata.json', pero falta el sidecar formal 'trips.metadata.json'; el artefacto no es válido para read_trips v1.1.", code='READ.LAYOUT.LEGACY_SIDECAR_DETECTED', details={'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_09_read_fatal_legacy_sidecar\\legacy_bundle.golondrina', 'resolved_path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_09_read_fatal_legacy_sidecar\\legacy_bundle.golondrina', 'legacy_file': 'metadata.json', 'expected_file': 'trips.metadata.json', 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.LEGACY_SIDECAR_DETECTED', message="Se detectó un sidecar legacy 'metadata.json', pero falta el sidecar formal 'trips.metadata.json'; el artefacto no es válido para read_trips v1.1.", field=None, source_field=None, row_count=None, details={'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07

OK - Test 9 - read fatal por sidecar legacy


### Test 10 - read degradado con recovery `strict=False`

Qué prueba:
- degradación controlada del sidecar;
- recuperación de lectura con warnings;
- `schema_effective` defaulted;
- `dataset_id` regenerado;
- `artifact_id=None`;
- `is_validated=False`.

In [22]:
case_dir = make_case_dir("test_10_read_degraded_recovery_strict_false")

written_trips, artifact_dir, _, _ = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="feather",
    feather_compression="lz4",
)

sidecar_path = artifact_dir / "trips.metadata.json"
payload = load_sidecar(artifact_dir)

payload.pop("schema_effective", None)
payload["dataset_id"] = ""
payload["artifact_id"] = None
payload["metadata"]["dataset_id"] = ""
payload["metadata"]["artifact_id"] = None
payload["metadata"]["is_validated"] = True

write_sidecar(artifact_dir, payload)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True

assert_issue_present(read_report.issues, "READ.SCHEMA_EFFECTIVE.DEFAULTED")
assert_issue_present(read_report.issues, "READ.METADATA.DATASET_ID_REGENERATED")
assert_issue_present(read_report.issues, "READ.METADATA.ARTIFACT_ID_SET_NONE")
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

assert loaded.metadata["is_validated"] is False
assert isinstance(loaded.metadata["dataset_id"], str) and loaded.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] is None

assert loaded.metadata["events"][-1]["op"] == "read_trips"

display(read_report.issues)
display(read_report.summary)
show_ok("Test 10 - read degradado con recovery strict=False")

[Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'missing_schema_effective_snapshot', 'strict': False, 'action': 'default_empty_schema_effective'}),
 Issue(level='warning', code='READ.METADATA.DATASET_ID_REGENERATED', message='El dataset_id persistido faltaba o era inválido; se regeneró un dataset_id efectivo para el dataset cargado.', field=None, source_field=None, row_count=None, details={'dataset_id': 'dset_5a9a9747-9ff5-47c6-b9b1-061ff0d40396', 'dataset_id_status': 'regenerated', 'previous_value': '', 'reason': 'missing_or_invalid_in_sidecar', 'action': 'regenerated'}),
 Issue(level='warning', code='READ.METADATA.ARTIFACT_ID_SET_NONE', message='El artifact_id persistido faltaba o era inválido; se dejará artifact_id=None en el dataset cargado.', field=None, source_field=None, row_count

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_10_read_degraded_recovery_strict_false\\bundle.golondrina',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_5a9a9747-9ff5-47c6-b9b1-061ff0d40396',
 'dataset_id_status': 'regenerated',
 'artifact_id': None,
 'artifact_id_status': 'missing_or_invalid'}

OK - Test 10 - read degradado con recovery strict=False


### Test 11 - read degradado fatal con `strict=True`

Qué prueba:
- el mismo tipo de degradación no debe recuperarse silenciosamente en modo estricto;
- ausencia de `schema_effective` debe abortar.

In [23]:
case_dir = make_case_dir("test_11_read_degraded_strict_true")

_, artifact_dir, _, _ = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="feather",
    feather_compression="lz4",
)

payload = load_sidecar(artifact_dir)
payload.pop("schema_effective", None)
write_sidecar(artifact_dir, payload)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=True,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.SCHEMA_EFFECTIVE.DEFAULTED"

display(raised)
show_ok("Test 11 - read degradado fatal con strict=True")

ExportError(message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', details={'reason': 'schema_effective unavailable or invalid in strict mode', 'strict': True, 'action': 'default_empty_schema_effective'}, issue=Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'schema_effective unavailable or invalid in strict mode', 'strict': True, 'action': 'default_empty_schema_effective'}), issues=(Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'schema_effective unavailable or invalid in strict mode', 

OK - Test 11 - read degradado fatal con strict=True


### Test 12 - política de `keep_metadata=False`

Qué prueba:
- `keep_metadata=False` no borra metadata ni provenance;
- solo evita append del evento `read_trips`;
- la regla post-read `is_validated=False` se mantiene.

In [24]:
case_dir = make_case_dir("test_12_read_keep_metadata_false")

written_trips, artifact_dir, _, _ = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="feather",
    feather_compression="lz4",
)

sidecar = load_sidecar(artifact_dir)
events_before = copy.deepcopy(sidecar["metadata"]["events"])

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=False,
    ),
)

assert read_report.ok is True
assert loaded.metadata["is_validated"] is False
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert ops_loaded == [ev["op"] for ev in events_before]
assert "read_trips" not in ops_loaded

assert "dataset_id" in loaded.metadata
assert loaded.provenance == written_trips.provenance

display(read_report.summary)
show_ok("Test 12 - política keep_metadata=False")

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_12_read_keep_metadata_false\\bundle.golondrina',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_e153cd4d-8b0a-4fc1-828c-034b447607d8',
 'artifact_id_status': 'loaded'}

OK - Test 12 - política keep_metadata=False


### Test 13 - coherencia sidecar / backend / data file en Feather

Qué prueba:
- el artefacto Feather debe ser autocontenible respecto del backend;
- `read_trips()` despacha por lo que declara el sidecar;
- `summary["storage_format"]` debe reflejar `feather`.

In [25]:
case_dir = make_case_dir("test_13_sidecar_backend_coherence_feather")

written_trips, artifact_dir, data_path, write_report = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="feather",
    feather_compression="lz4",
)

sidecar = load_sidecar(artifact_dir)

assert sidecar["storage"]["format"] == "feather"
assert sidecar["files"]["data"] == "trips.feather"
assert data_path.name == "trips.feather"
assert write_report.summary["storage_format"] == "feather"

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True
assert read_report.summary["storage_format"] == "feather"
assert loaded.metadata["artifact_id"] == written_trips.metadata["artifact_id"]

display(sidecar["storage"])
display(read_report.summary)
show_ok("Test 13 - coherencia sidecar/backend/data file Feather")

{'format': 'feather', 'options': {'compression': 'lz4', 'version': 2}}

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_13_sidecar_backend_coherence_feather\\bundle.golondrina',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_28951706-1067-4b4c-a9c5-c7966c47ded7',
 'artifact_id_status': 'loaded'}

OK - Test 13 - coherencia sidecar/backend/data file Feather


### Test 14 - mismatch entre `storage.format` y `files.data`

Qué prueba:
- endurecimiento del preflight de lectura;
- si `storage.format="feather"` pero `files.data="trips.parquet"`, el artefacto es incoherente y debe fallar.

In [26]:
case_dir = make_case_dir("test_14_data_file_mismatch")

_, artifact_dir, _, _ = write_valid_artifact_with_backend(
    case_dir,
    artifact_name="bundle",
    storage_format="feather",
    feather_compression="lz4",
)

payload = load_sidecar(artifact_dir)
payload["storage"]["format"] = "feather"
payload["files"]["data"] = "trips.parquet"
write_sidecar(artifact_dir, payload)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ.LAYOUT.DATA_FILE_MISMATCH"

display(raised)
show_ok("Test 14 - mismatch storage.format/files.data")

ExportError(message="El sidecar declara un archivo de datos 'trips.parquet' inconsistente con storage.format='feather'; se esperaba 'trips.feather'.", code='READ.LAYOUT.DATA_FILE_MISMATCH', details={'expected_file': 'trips.feather', 'declared_file': 'trips.parquet', 'storage_format': 'feather', 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.DATA_FILE_MISMATCH', message="El sidecar declara un archivo de datos 'trips.parquet' inconsistente con storage.format='feather'; se esperaba 'trips.feather'.", field=None, source_field=None, row_count=None, details={'expected_file': 'trips.feather', 'declared_file': 'trips.parquet', 'storage_format': 'feather', 'action': 'abort'}), issues=(Issue(level='error', code='READ.LAYOUT.DATA_FILE_MISMATCH', message="El sidecar declara un archivo de datos 'trips.parquet' inconsistente con storage.format='feather'; se esperaba 'trips.feather'.", field=None, source_field=None, row_count=None, details={'expected_file': 'trips.feather', 'declare

OK - Test 14 - mismatch storage.format/files.data


### Test 15 - integridad lógica de columnas categóricas tras roundtrip Feather

Qué prueba:
- preservación fila a fila de valores observados;
- preservación del patrón de nulos;
- preservación de conteos por categoría observada;
- preservación del conjunto de categorías observadas.

No exige preservar exactamente el dtype pandas, porque la frontera contractual relevante del read es el contenido lógico reconstruido.

In [27]:
case_dir = make_case_dir("test_15_categorical_integrity_roundtrip_feather")

trips_original = clone_tripdataset(tripdataset_validated_small)

write_report = write_trips(
    trips_original,
    case_dir / "integrity_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=True,
    ),
)

loaded, read_report = read_trips(
    case_dir / "integrity_bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

checked_cols = []

for col in selected_categorical_columns(trips_original.data):
    if col not in loaded.data.columns:
        continue

    checked_cols.append(col)

    orig_s = series_as_string_with_na(trips_original.data[col]).reset_index(drop=True)
    load_s = series_as_string_with_na(loaded.data[col]).reset_index(drop=True)

    pd.testing.assert_series_equal(
        load_s,
        orig_s,
        check_names=False,
        check_dtype=False,
    )

    pd.testing.assert_series_equal(
        loaded.data[col].isna().reset_index(drop=True),
        trips_original.data[col].isna().reset_index(drop=True),
        check_names=False,
    )

    orig_counts = orig_s.value_counts(dropna=False).sort_index()
    load_counts = load_s.value_counts(dropna=False).sort_index()

    pd.testing.assert_series_equal(
        load_counts,
        orig_counts,
        check_names=False,
        check_dtype=False,
    )

    assert observed_non_null_values(loaded.data[col]) == observed_non_null_values(trips_original.data[col])

assert checked_cols, "No se encontró ninguna columna categórica observable para verificar."

display({"checked_cols": checked_cols})
display(read_report.summary)
show_ok("Test 15 - integridad lógica categórica tras roundtrip Feather")

{'checked_cols': ['mode',
  'purpose',
  'day_type',
  'time_period',
  'user_gender',
  'user_age_group',
  'income_quintile']}

{'n_rows': 180,
 'n_columns': 33,
 'path': 'C:\\projects\\pylondrina\\notebooks\\testing\\io_trips\\tmp_op07_read_trips_integration\\test_15_categorical_integrity_roundtrip_feather\\integrity_bundle.golondrina',
 'storage_format': 'feather',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_4ee3659f1e1942b39477cca1eda8bb47',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_52e4d07b-2e04-4f5b-93e2-fdd7a26e12fe',
 'artifact_id_status': 'loaded'}

OK - Test 15 - integridad lógica categórica tras roundtrip Feather
